# 06. Volume and spread prediction

Two auxiliary deep-learning experiments documented in Section 9 of the report.

The deployed deep-learning pipeline (Section 6.3) predicts only the mid-price trajectory. Two natural extensions are predicting per-second volume and per-second spread, both with the same DeepLOB + BiLSTM encoder. The motivation:

- **Volume forecast** would let a VWAP-style scheduler concentrate volume in seconds expected to be high-volume, so each unit walks shallower into the book.
- **Spread forecast** would let the scheduler avoid seconds with wide spreads.

Both experiments share the encoder with the deployed pipeline but differ in the target and decoder head. This notebook contains complete, runnable pipelines for both: data loading, preprocessing, training, evaluation, and the canonical R^2 numbers.

Training requires PyTorch and a GPU (typically a single Colab T4 is enough).

## 0. Setup

Standard imports plus the canonical SHA-1 holdout. Adjust `SYMBOL` to evaluate on any of the seven pairs.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from execution_edge.splits import compute_holdout_partition, per_symbol_split
from execution_edge.models.deeplob import DeepLOBEncoder
from execution_edge.models.multi_head import DeepLOBForecastMulti, multi_head_loss, TARGET_NAMES
from execution_edge.data import (
    DATA_DIR, ASK_PRICE_COLS, ASK_VOL_COLS, BID_PRICE_COLS, BID_VOL_COLS,
    load_volume_to_fill,
)

SYMBOLS = ["BTCUSDT","ETHUSDT","LTCUSDT","SOLUSDT","ADAUSDT","DOGEUSDT","XRPUSDT"]
SYMBOL = "BTCUSDT"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

dev_ids, holdout_ids = compute_holdout_partition(DATA_DIR, SYMBOLS, fraction=0.20)
print(f"holdout pool: {len(holdout_ids)} ids")

## 1. Volume prediction

The model is identical to the deployed mid-price pipeline (DeepLOB + BiLSTM) but the decoder is replaced by a non-autoregressive linear head that emits 60 per-second volume forecasts in one shot. The target is the per-second OHLCV trade volume on the last minute, log-transformed (`np.log1p`) to compress its heavy tail and then z-scored per instrument.

In [ ]:
class VolumeModel(nn.Module):
    """DeepLOB + BiLSTM + non-autoregressive head for per-second volume."""

    def __init__(self, hidden: int = 128, horizon: int = 60, dropout: float = 0.1):
        super().__init__()
        self.spatial = DeepLOBEncoder(in_ch=1)
        self.encoder = nn.LSTM(
            input_size=192, hidden_size=hidden,
            num_layers=1, batch_first=True, bidirectional=True,
        )
        self.enc_dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(2 * hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, horizon),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.spatial(x[:, :, :20])
        enc_out, _ = self.encoder(h)
        enc_out = self.enc_dropout(enc_out)
        pooled = enc_out.mean(dim=1)
        return self.head(pooled)

### Data loading and preprocessing

In [ ]:
LOB_COLS = []
for i in range(1, 6):
    LOB_COLS.extend([f"ask_price_{i}", f"ask_vol_{i}", f"bid_price_{i}", f"bid_vol_{i}"])

INPUT_WINDOW = 600
TARGET_WINDOW = 60


def load_pair_data(symbol: str):
    x = pd.read_parquet(DATA_DIR / symbol / "X_train.parquet").sort_values(
        ["anonymized_id", "time_in_hour"]).reset_index(drop=True)
    y = pd.read_parquet(DATA_DIR / symbol / "y_train.parquet").sort_values(
        ["anonymized_id", "time_in_hour"]).reset_index(drop=True)
    return x, y


def split_into_dev_holdout(x: pd.DataFrame, y: pd.DataFrame, holdout_ids: set):
    ids = np.sort(x["anonymized_id"].unique())
    dev_ids_sym, held_ids_sym = per_symbol_split(ids.tolist(), holdout_ids)
    dev_x = x[x["anonymized_id"].isin(dev_ids_sym)].reset_index(drop=True)
    dev_y = y[y["anonymized_id"].isin(dev_ids_sym)].reset_index(drop=True)
    held_x = x[x["anonymized_id"].isin(held_ids_sym)].reset_index(drop=True)
    held_y = y[y["anonymized_id"].isin(held_ids_sym)].reset_index(drop=True)
    return dev_x, dev_y, held_x, held_y


def log1p_volume(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["volume"] = np.log1p(df["volume"].clip(lower=0).fillna(0))
    return df


class VolumeDataset(Dataset):
    """One sample per anonymized_id: the visible window x and the next-minute log-volume target."""

    def __init__(self, x_df: pd.DataFrame, y_df: pd.DataFrame, feat_means: np.ndarray, feat_stds: np.ndarray,
                 vol_mean: float, vol_std: float):
        self.samples = []
        ids = x_df["anonymized_id"].unique()
        for hid in ids:
            xh = x_df[x_df["anonymized_id"] == hid].sort_values("time_in_hour").tail(INPUT_WINDOW)
            yh = y_df[y_df["anonymized_id"] == hid].sort_values("time_in_hour")
            if len(yh) != TARGET_WINDOW:
                continue
            x_arr = xh[LOB_COLS].astype(np.float32).to_numpy()
            if x_arr.shape[0] < INPUT_WINDOW:
                pad = np.zeros((INPUT_WINDOW - x_arr.shape[0], x_arr.shape[1]), dtype=np.float32)
                x_arr = np.vstack([pad, x_arr])
            x_arr = (x_arr - feat_means) / feat_stds
            y_log = np.log1p(yh["volume"].clip(lower=0).fillna(0).to_numpy(dtype=np.float32))
            y_z = (y_log - vol_mean) / vol_std
            self.samples.append((x_arr, y_z, hid))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y, hid = self.samples[idx]
        return torch.from_numpy(x), torch.from_numpy(y)

### Training driver

In [ ]:
def train_volume_pipeline(symbol: str = SYMBOL, epochs: int = 50, batch_size: int = 32, lr: float = 1e-3):
    """Train one VolumeModel on the symbol's dev partition; return model + scalers + val loader."""
    x, y = load_pair_data(symbol)
    dev_x, dev_y, held_x, held_y = split_into_dev_holdout(x, y, holdout_ids)

    # Per-feature stats from dev only (no leakage from holdout).
    feat_means = dev_x[LOB_COLS].mean().to_numpy().astype(np.float32)
    feat_stds = dev_x[LOB_COLS].std().replace(0, 1e-6).to_numpy().astype(np.float32)

    dev_y_log = log1p_volume(dev_y)
    vol_mean = float(dev_y_log["volume"].mean())
    vol_std = float(dev_y_log["volume"].std() or 1e-6)

    train_ds = VolumeDataset(dev_x, dev_y, feat_means, feat_stds, vol_mean, vol_std)
    held_ds = VolumeDataset(held_x, held_y, feat_means, feat_stds, vol_mean, vol_std)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    held_loader = DataLoader(held_ds, batch_size=batch_size, shuffle=False)

    model = VolumeModel().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    for epoch in range(epochs):
        model.train(); total, n = 0.0, 0
        for xb, yb in train_loader:
            xb = xb.to(device); yb = yb.to(device)
            opt.zero_grad()
            pred = model(xb)
            loss = nn.functional.huber_loss(pred, yb, delta=1.0)
            loss.backward(); opt.step()
            total += float(loss.item()) * xb.size(0); n += xb.size(0)
        print(f"epoch {epoch+1}: train loss {total/n:.4f}")

    scalers = {"feat_means": feat_means, "feat_stds": feat_stds,
               "vol_mean": vol_mean, "vol_std": vol_std}
    return model, scalers, held_loader, held_x, held_y


# Run when you have a GPU available:
# model, scalers, held_loader, held_x, held_y = train_volume_pipeline(symbol="BTCUSDT", epochs=50)

### Evaluation: R^2 on holdout

Predictions come back in z-scored log space. To compute R^2 in raw volume space, undo the z-score, then `expm1` to recover raw volume.

In [ ]:
def evaluate_volume(model, scalers, held_loader):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for xb, yb in held_loader:
            xb = xb.to(device); yb = yb.to(device)
            pred = model(xb).cpu()
            all_preds.append(pred); all_targets.append(yb.cpu())
    P = torch.cat(all_preds)        # [N, 60] z-scored log-volume
    T = torch.cat(all_targets)

    # Normalised R^2 (in z-score space, the "honest" metric).
    pooled_mean = T.mean()
    ss_tot = ((T - pooled_mean) ** 2).sum().item()
    ss_res = ((T - P) ** 2).sum().item()
    r2_normalized = 1 - ss_res / max(ss_tot, 1e-12)

    # Raw-volume R^2 (after denormalising).
    P_raw = torch.expm1(P * scalers["vol_std"] + scalers["vol_mean"])
    T_raw = torch.expm1(T * scalers["vol_std"] + scalers["vol_mean"])
    pooled_raw_mean = T_raw.mean()
    ss_tot_raw = ((T_raw - pooled_raw_mean) ** 2).sum().item()
    ss_res_raw = ((T_raw - P_raw) ** 2).sum().item()
    r2_denormalized = 1 - ss_res_raw / max(ss_tot_raw, 1e-12)

    # Per-instrument R^2 distribution.
    per_inst_ss_res = ((T - P) ** 2).sum(dim=1)
    per_inst_ss_tot = ((T - T.mean(dim=1, keepdim=True)) ** 2).sum(dim=1)
    per_inst_r2 = 1 - per_inst_ss_res / per_inst_ss_tot.clamp_min(1e-12)
    valid = per_inst_ss_tot > 1e-6
    return {
        "r2_normalized": r2_normalized,
        "r2_denormalized": r2_denormalized,
        "median_per_inst_r2": per_inst_r2[valid].median().item(),
        "frac_positive_r2": (per_inst_r2[valid] > 0).float().mean().item(),
        "n_valid": int(valid.sum().item()),
    }


# Canonical run, BTC volume on the SHA-1 holdout (run on GPU):
# metrics = evaluate_volume(model, scalers, held_loader)
# Expected pooled normalised R^2: about -2.2 (the figure cited in the report).

## 2. Spread prediction (multi-target model)

The spread experiment uses a multi-target predictor that emits three trajectories simultaneously: mid-price, spread, and top-of-book liquidity. The architecture is `DeepLOBForecastMulti` (defined in `execution_edge/models/multi_head.py`). Three heads share the same encoder, then a single linear layer produces 60 seconds times three targets in one shot.

The schedule then combines all three forecasts: price favourability against the predicted close, plus a liquidity tilt, blended with last-K TWAP. Spread is included in the model but the report notes it was too unstable in this auxiliary experiment to use as a scheduler input.

In [ ]:
def compute_target_scalers(dev_y: pd.DataFrame) -> dict:
    """Per-target mean/std from dev only; used to normalise the three heads."""
    mid = (dev_y["ask_price_1"] + dev_y["bid_price_1"]) / 2.0
    spread = dev_y["ask_price_1"] - dev_y["bid_price_1"]
    liq = dev_y["ask_vol_1"] + dev_y["bid_vol_1"]
    return {
        "mid_mean": float(mid.mean()),    "mid_std": float(mid.std() or 1e-6),
        "spread_mean": float(spread.mean()),"spread_std": float(spread.std() or 1e-6),
        "liq_mean": float(liq.mean()),    "liq_std": float(liq.std() or 1e-6),
    }


class MultiTargetDataset(Dataset):
    """Builds (x_window, [mid_z, spread_z, liq_z]) samples per anonymized_id."""

    def __init__(self, x_df, y_df, feat_means, feat_stds, target_scalers):
        self.samples = []
        ids = x_df["anonymized_id"].unique()
        for hid in ids:
            xh = x_df[x_df["anonymized_id"] == hid].sort_values("time_in_hour").tail(INPUT_WINDOW)
            yh = y_df[y_df["anonymized_id"] == hid].sort_values("time_in_hour")
            if len(yh) != TARGET_WINDOW: continue
            x_arr = xh[LOB_COLS].astype(np.float32).to_numpy()
            if x_arr.shape[0] < INPUT_WINDOW:
                pad = np.zeros((INPUT_WINDOW - x_arr.shape[0], x_arr.shape[1]), dtype=np.float32)
                x_arr = np.vstack([pad, x_arr])
            x_arr = (x_arr - feat_means) / feat_stds
            mid = (yh["ask_price_1"].to_numpy(np.float32) + yh["bid_price_1"].to_numpy(np.float32)) / 2.0
            spread = yh["ask_price_1"].to_numpy(np.float32) - yh["bid_price_1"].to_numpy(np.float32)
            liq = yh["ask_vol_1"].to_numpy(np.float32) + yh["bid_vol_1"].to_numpy(np.float32)
            y = np.stack([
                (mid - target_scalers["mid_mean"]) / target_scalers["mid_std"],
                (spread - target_scalers["spread_mean"]) / target_scalers["spread_std"],
                (liq - target_scalers["liq_mean"]) / target_scalers["liq_std"],
            ], axis=1)  # [60, 3]
            self.samples.append((x_arr, y, hid))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y, _ = self.samples[idx]
        return torch.from_numpy(x), torch.from_numpy(y)

In [ ]:
def train_spread_pipeline(symbol: str = SYMBOL, epochs: int = 50, batch_size: int = 16, lr: float = 1e-3):
    x, y = load_pair_data(symbol)
    dev_x, dev_y, held_x, held_y = split_into_dev_holdout(x, y, holdout_ids)

    feat_means = dev_x[LOB_COLS].mean().to_numpy().astype(np.float32)
    feat_stds = dev_x[LOB_COLS].std().replace(0, 1e-6).to_numpy().astype(np.float32)
    target_scalers = compute_target_scalers(dev_y)

    train_ds = MultiTargetDataset(dev_x, dev_y, feat_means, feat_stds, target_scalers)
    held_ds = MultiTargetDataset(held_x, held_y, feat_means, feat_stds, target_scalers)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    held_loader = DataLoader(held_ds, batch_size=batch_size, shuffle=False)

    model = DeepLOBForecastMulti().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    for epoch in range(epochs):
        model.train(); total, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            pred = model(xb)
            loss = multi_head_loss(pred, yb, head_weights=(1.0, 0.2, 0.2), smooth_lambda=0.02)
            loss.backward(); opt.step()
            total += float(loss.item()) * xb.size(0); n += xb.size(0)
        print(f"epoch {epoch+1}: train loss {total/n:.4f}")

    scalers = {"feat_means": feat_means, "feat_stds": feat_stds, **target_scalers}
    return model, scalers, held_loader, held_x, held_y


def evaluate_multi(model, scalers, held_loader):
    """Per-head R^2 on the holdout (z-scored space)."""
    model.eval()
    all_p, all_t = [], []
    with torch.no_grad():
        for xb, yb in held_loader:
            xb, yb = xb.to(device), yb.to(device)
            all_p.append(model(xb).cpu()); all_t.append(yb.cpu())
    P = torch.cat(all_p)  # [N, 60, 3]
    T = torch.cat(all_t)
    results = {}
    for h, name in enumerate(TARGET_NAMES):
        ph, th = P[:, :, h], T[:, :, h]
        mean = th.mean()
        ss_tot = ((th - mean) ** 2).sum().item()
        ss_res = ((th - ph) ** 2).sum().item()
        results[f"{name}_R2_normalized"] = 1 - ss_res / max(ss_tot, 1e-12)
    return results


# Canonical run, BTC multi-head on the SHA-1 holdout (run on GPU):
# model_m, scalers_m, held_loader_m, _, _ = train_spread_pipeline(symbol="BTCUSDT", epochs=50)
# metrics_m = evaluate_multi(model_m, scalers_m, held_loader_m)
# Expected: mid R^2 around 0.86, spread R^2 unstable (negative on some pairs), liq R^2 weakly positive.

## 3. Canonical R^2 numbers (cached)

These are the numbers from the canonical training runs. To regenerate them, run the training cells above on a machine with GPU (about 30 minutes per pair for the volume model, similar for the multi-head model).

In [1]:
import pandas as pd

volume_metrics = pd.DataFrame({
    "Metric":                        ["Pooled normalised R^2",
                                      "Pooled denormalised R^2",
                                      "Median per-instrument R^2 (normalised)",
                                      "Fraction of instruments with R^2 > 0",
                                      "Reference: mid-price normalised R^2 (BTC)"],
    "Value (DeepLOB volume, BTC)":   [-2.23, -2.26, -0.036, 0.078, 0.86],
}).set_index("Metric")
volume_metrics

                                            Value (DeepLOB volume, BTC)
Metric                                                                  
Pooled normalised R^2                                            -2.230
Pooled denormalised R^2                                          -2.260
Median per-instrument R^2 (normalised)                           -0.036
Fraction of instruments with R^2 > 0                              0.078
Reference: mid-price normalised R^2 (BTC)                         0.860

## 4. Takeaway

Both experiments produced negative per-instrument R^2 in z-score space and are documented in Section 9 of the report. The per-second time scale is below the horizon at which volume and spread carry forecastable structure: the literature only reports positive volume R^2 at fifteen-to-twenty-minute horizons where the intraday U-shape and the closing ramp dominate the variance. Pushing the horizon down to one second strips out those seasonal components and leaves a per-second trade count dominated by the random arrival of individual orders.

The adaptive scheduling pipeline of Section 6.4 addresses the same problem from the other side: instead of trying to predict the per-second profile, it conditions on hour-level summaries that are coarse enough to be reliably estimated, and beats TWAP-K on six of seven ask-side pairs.